In [ ]:
!pip install -q trl bitsandbytes "torchao>=0.16.0" "transformers<5.0"

## Setup

Install deps, authenticate with Hugging Face, load the tokenizer, and validate the SFT dataset.
The two flags below (`TRAINING`, `EVALUATION`) gate everything after this section.

In [ ]:
import contextlib
import csv
import difflib
import functools
import json
import math
import os
import re
import string
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import create_repo, login, upload_folder
from kaggle_secrets import UserSecretsClient
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback,
)
from trl import SFTConfig, SFTTrainer

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
DATA_PATH = "/kaggle/input/datasets/jopopapi/pii-stf-dataset/sft_dataset.jsonl"

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN_W")
login(HF_TOKEN)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def validate_sft_jsonl(path: str) -> None:
    """Print a warning for any line that is not a dict with a list-valued 'messages' key."""
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            try:
                record = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"Line {i} failed to parse: {e}")
                continue
            if not isinstance(record, dict):
                print(f"Line {i} is not a dict (type={type(record)})")
            elif "messages" not in record:
                print(f"Line {i} missing 'messages'")
            elif not isinstance(record["messages"], list):
                print(f"Line {i} 'messages' is not a list (type={type(record['messages'])})")


if os.path.exists(DATA_PATH):
    validate_sft_jsonl(DATA_PATH)
else:
    print(f"DATA_PATH {DATA_PATH!r} not found -- skipping dataset validation (eval-only session).")

## Training

QLoRA fine-tune of the base model on the chat-formatted SFT dataset. Every cell below is gated on `TRAINING`; jump to **Evaluation** to score an existing adapter instead.

In [ ]:
# Toggle each stage independently: TRAINING runs the fine-tune,
# EVALUATION runs the redaction harness + scoring. Either, both, or neither.
TRAINING = False
EVALUATION = True

In [ ]:
if TRAINING:
    dataset = load_dataset(
        "json",
        data_files=DATA_PATH,
        split="train"
    )

In [ ]:
if TRAINING:

    def format_chat(example):
        """Render one chat example to a single training string via the Llama chat template."""
        return {
            "text": tokenizer.apply_chat_template(
                example["messages"], tokenize=False, add_generation_prompt=False
            )
        }

    def compute_len(example):
        """Token length of the rendered 'text' field."""
        return {"len": len(tokenizer(example["text"])["input_ids"])}

In [ ]:
if TRAINING:
    dataset = dataset.map(format_chat, remove_columns=dataset.column_names)
    lengths = dataset.map(compute_len)
    dataset = dataset.train_test_split(test_size=0.05, seed=42)

In [ ]:
if TRAINING:
    all_lens = lengths["len"]
    print(f"Mean token length: {sum(all_lens) / len(all_lens):.2f}")
    print(f"Max token length: {max(all_lens)}")
    print(f"Min token length: {min(all_lens)}")

In [ ]:
if TRAINING:

    def compute_lora_norms(model):
        """Total (and per-tensor) L2 norm of trainable LoRA weights -- proxy for adapter movement."""
        norms = {}
        total_norm = 0.0
        for name, param in model.named_parameters():
            if "lora_" in name and param.requires_grad:
                norm = torch.norm(param.detach()).item()
                norms[name] = norm
                total_norm += norm ** 2
        return total_norm ** 0.5, norms

In [ ]:
if TRAINING:
    class MetricsCallback(TrainerCallback):
        def __init__(self):
            self.train_losses = []
            self.eval_losses = []
            self.perplexities = []
            self.lora_norms = []
            self.steps = []
    
        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs is None:
                return
    
            if "loss" in logs:
                self.train_losses.append(logs["loss"])
                self.steps.append(state.global_step)
    
        def on_evaluate(self, args, state, control, metrics=None, **kwargs):
            if metrics is None:
                return
    
            eval_loss = metrics.get("eval_loss")
            if eval_loss is not None:
                self.eval_losses.append(eval_loss)
                self.perplexities.append(math.exp(eval_loss))
    
            model = kwargs["model"]
            lora_norm, _ = compute_lora_norms(model)
            self.lora_norms.append(lora_norm)

In [ ]:
if TRAINING:
    print("Downloading model")

    # QLoRA 4-bit NF4: keeps an 8B base under ~5 GB so it fits one T4/P100 without device_map sharding.
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
    )
    base.config.use_cache = False

In [ ]:
if TRAINING:
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=False)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )

    model = get_peft_model(base, lora_config)
    model.enable_input_require_grads()
    model.print_trainable_parameters()

    metrics_callback = MetricsCallback()

In [ ]:
if TRAINING:
    print("training started")

    # SMOKE_TEST=True: train a tiny slice for 3 steps to surface hangs / OOM before a full run.
    SMOKE_TEST = False
    train_subset = dataset["train"].select(range(min(32, len(dataset["train"])))) if SMOKE_TEST else dataset["train"]
    eval_subset = dataset["test"].select(range(min(8, len(dataset["test"])))) if SMOKE_TEST else dataset["test"]

    NUM_TRAIN_EPOCHS = 2
    PER_DEVICE_TRAIN_BATCH_SIZE = 4
    GRADIENT_ACCUMULATION_STEPS = 4

    effective_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
    steps_per_epoch = math.ceil(len(train_subset) / effective_batch_size)
    total_steps = steps_per_epoch * NUM_TRAIN_EPOCHS
    eval_save_steps = 2 if SMOKE_TEST else max(1, total_steps // 10)
    print(f"steps_per_epoch={steps_per_epoch} total_steps={total_steps} eval_save_steps={eval_save_steps}")

    training_args = SFTConfig(
        output_dir="/kaggle/working/llama31-pii/checkpoints",
        loss_type="nll",
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=1e-4,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        max_steps=3 if SMOKE_TEST else -1,
        warmup_ratio=0.05,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        optim="adamw_torch",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=1 if SMOKE_TEST else max(1, eval_save_steps // 2),
        eval_strategy="steps",
        eval_steps=eval_save_steps,
        save_strategy="steps",
        save_steps=eval_save_steps,
        save_total_limit=3,
        ddp_find_unused_parameters=False,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",
    )
    print("SFTConfig built")

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_subset,
        eval_dataset=eval_subset,
        args=training_args,
        callbacks=[metrics_callback],
    )
    print("SFTTrainer built, calling train()")

    trainer.train()
    print("train() returned")
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated(0) / 1e9:.2f} GB")
    print(
        f"metrics_callback: {len(metrics_callback.train_losses)} train-loss points, "
        f"{len(metrics_callback.eval_losses)} eval-loss points, "
        f"{len(metrics_callback.lora_norms)} LoRA-norm points"
    )

In [ ]:
if TRAINING:
    _curves = [
        ("Training Loss", "Steps", metrics_callback.steps, metrics_callback.train_losses),
        ("Validation Loss", "Eval Step", None, metrics_callback.eval_losses),
        ("Perplexity", "Eval Step", None, metrics_callback.perplexities),
        ("LoRA Weight Norm", "Eval Step", None, metrics_callback.lora_norms),
    ]
    for title, xlabel, xs, ys in _curves:
        plt.figure()
        if xs is None:
            plt.plot(ys)
        else:
            plt.plot(xs, ys)
        plt.xlabel(xlabel)
        plt.ylabel(title)
        plt.title(title)
        plt.show()

In [ ]:
if TRAINING:
    final_dir = "/kaggle/working/llama31-pii/final"
    trainer.model.save_pretrained(final_dir)
    tokenizer.save_pretrained(final_dir)

In [ ]:
if TRAINING:
    model_name_for_repo = MODEL_NAME.split("/")[-1]
    OUTPUT_REPO = f"jiminaa/{model_name_for_repo}-PII-LoRA"

    try:
        create_repo(repo_id=OUTPUT_REPO, exist_ok=True, repo_type="model", token=HF_TOKEN)
        print(f"Repository ready: {OUTPUT_REPO}")
    except Exception as e:
        print(f"Note: {e}")

    try:
        upload_folder(folder_path=final_dir, repo_id=OUTPUT_REPO, repo_type="model", token=HF_TOKEN)
    except Exception as e:
        print(f"Note: {e}")

## Evaluation

Set `EVALUATION = True` in the flags cell. This runs without training: every adapter in
`ADAPTER_SPECS` loads from the Hub repo (plus the freshly trained checkpoint if one exists in
this session), and the base model is always included as a reference point.

The pipeline mirrors `scripts/de_identify_steplab.py` (redaction harness) and
`scripts/transcript_evaluator.py` (token-level scoring), swapping the OpenRouter call for a
local `.generate()`.

In [ ]:
if EVALUATION:
    # Adapters to evaluate against the base model: (label, subfolder) pairs, all under one
    # HF repo -- each subfolder holds one adapter's adapter_config.json + weights. Subfolder
    # names encode hyperparams as "{learning_rate}_{num_train_epochs}_{lora_r}".
    ADAPTER_REPO = "jiminaa/Llama-3.1-8B-Instruct-PII-LoRA"
    ADAPTER_SPECS = [(name, name) for name in [
        "1e-4_2_16",
        "1e-4_3_4",
        "2e-4_3_16",
        "2e-4_3_4",
        "2e-4_3_8",
        "2e-4_8_4",
        "4e-4_3_4",
        "5e-5_3_4",
        "8e-4_3_4",
        "kaggle"
    ]]

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    sanity_base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map={"": 0}
    )

    # Every adapter is loaded onto this one quantized base as a named PEFT adapter, rather
    # than re-downloading/re-quantizing the base per adapter. If this session just trained a
    # checkpoint, it's evaluated too, alongside the repo's adapters.
    adapter_sources = []
    _local_final_dir = globals().get("final_dir")
    if _local_final_dir and os.path.isfile(os.path.join(_local_final_dir, "adapter_config.json")):
        adapter_sources.append(("local", _local_final_dir, None))
    adapter_sources.extend((label, ADAPTER_REPO, subfolder) for label, subfolder in ADAPTER_SPECS)

    sanity_model = None
    adapter_labels = []
    for label, path, subfolder in adapter_sources:
        kwargs = {"subfolder": subfolder} if subfolder else {}
        if sanity_model is None:
            sanity_model = PeftModel.from_pretrained(sanity_base, path, adapter_name=label, **kwargs)
        else:
            sanity_model.load_adapter(path, adapter_name=label, **kwargs)
        adapter_labels.append(label)
        suffix = f" (subfolder={subfolder!r})" if subfolder else ""
        print(f"Loaded adapter '{label}' from {path}{suffix}")

    @contextlib.contextmanager
    def use_adapter(model, name):
        """Temporarily make `name` the active adapter on `model` for a `with` block."""
        model.set_adapter(name)
        yield

    if sanity_model is not None:
        sanity_model.eval()
    else:
        sanity_base.eval()

    demo_model = sanity_model if sanity_model is not None else sanity_base

    # Quick smoke test: redact the first training example and print the model's output.
    if os.path.exists(DATA_PATH):
        eval_example = load_dataset("json", data_files=DATA_PATH, split="train")[0]["messages"][:-1]
        prompt = tokenizer.apply_chat_template(eval_example, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(demo_model.device)
        with torch.no_grad():
            output_ids = demo_model.generate(**inputs, max_new_tokens=256, do_sample=False)
        print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    else:
        print(f"DATA_PATH {DATA_PATH!r} not found -- skipping single-example demo generation.")

### Redaction pipeline

Helpers ported from `scripts/de_identify_steplab.py`: text normalization, token-level validation, output alignment repair, and the tiered chunk → line → word fallback.

In [ ]:
if EVALUATION:
    # Text normalization + speaker formatting, ported from scripts/de_identify_steplab.py.

    _PUNCT = set(string.punctuation)
    REDACT_PATTERN = re.compile(r"\[REDACTED\]", re.I)
    _SPEAKER_LINE_RE = re.compile(r"^\s*Speaker\s*[A-Za-z0-9]*\s*:")
    _EMPTY_SPEAKER_LINE_RE = re.compile(r"^Speaker\s*[A-Za-z0-9]*\s*:\s*$")
    _SPEAKER_SPLIT_RE = re.compile(r"(Speaker\s*[A-Za-z0-9]*\s*:)")

    def _is_redacted_with_punct(tok: str) -> bool:
        """True if `tok` is '[REDACTED]' bar surrounding punctuation and an optional 's possessive."""
        if not tok:
            return False
        idx = tok.find("[REDACTED]")
        if idx == -1:
            return False
        before = tok[:idx]
        after = tok[idx + len("[REDACTED]"):]
        if after.startswith(("'s", "’s")):
            after = after[2:]
        return all(c in _PUNCT for c in before) and all(c in _PUNCT for c in after)

    def normalize_whitespace(text: str) -> str:
        """Collapse runs of whitespace within each line, keeping line breaks."""
        return "\n".join(re.sub(r"\s+", " ", line.strip()) for line in text.split("\n"))

    def _is_speaker_line(line: str) -> bool:
        return bool(_SPEAKER_LINE_RE.match(line))

    def normalize_speaker_format(text: str) -> str:
        """Put each 'Speaker X:' turn on its own paragraph."""
        parts = _SPEAKER_SPLIT_RE.split(normalize_whitespace(text))
        turns: list[str] = []
        current_speaker = None
        for part in parts:
            part = part.strip()
            if not part:
                continue
            if _SPEAKER_SPLIT_RE.match(part):
                current_speaker = part
            elif current_speaker:
                turns.append(f"{current_speaker} {part}")
                current_speaker = None
            else:
                turns.append(part)
        return "\n\n".join(turns)

    def remove_empty_speaker_lines(text: str) -> str:
        """Drop blank lines and 'Speaker X:' lines that have no content."""
        return "\n".join(
            stripped
            for line in text.split("\n")
            if (stripped := line.strip()) and not _EMPTY_SPEAKER_LINE_RE.match(stripped)
        )

    def split_into_sentences(text: str, max_sentences: int = 3) -> list[str]:
        """Group text into chunks of up to `max_sentences` sentences (matches build_sft_dataset.py)."""
        sentences = re.split(r"(?<=[.!?])\s+", text)
        return [
            " ".join(sentences[i:i + max_sentences])
            for i in range(0, len(sentences), max_sentences)
        ]

    def find_pattern_in_response(pattern, response_words) -> int:
        """First index where `pattern` occurs in `response_words`, treating [REDACTED] as a wildcard."""
        if len(pattern) > len(response_words):
            return -1
        for start in range(len(response_words) - len(pattern) + 1):
            if all(
                response_words[start + i] == pattern[i]
                or _is_redacted_with_punct(response_words[start + i])
                for i in range(len(pattern))
            ):
                return start
        return -1

In [ ]:
if EVALUATION:
    # Token-level validation, ported from scripts/de_identify_steplab.py.

    _TOKEN_RE = re.compile(r"\[REDACTED\](?:'s|’s)?|[A-Za-z0-9']+")
    _REFUSAL_MARKERS = (
        "please provide the text",
        "there's no actual text",
        "personally identifiable information (pii)",
    )

    def _normalize_smart_punct(s: str) -> str:
        """Fold curly quotes, dashes, and non-breaking spaces to ASCII equivalents."""
        s = unicodedata.normalize("NFC", s)
        return (
            s.replace("’", "'").replace("‘", "'").replace("ʼ", "'").replace("‛", "'")
             .replace("“", '"').replace("”", '"').replace("‟", '"')
             .replace("–", "-").replace("—", "-").replace("‒", "-")
             .replace(" ", " ").replace(" ", " ").replace(" ", " ")
        )

    def _word_tokens(s: str) -> list[str]:
        """Word / [REDACTED] tokens, punctuation dropped, every redaction variant folded to '[REDACTED]'."""
        return [
            "[REDACTED]" if t.startswith("[REDACTED]") else t
            for t in _TOKEN_RE.findall(_normalize_smart_punct(s))
        ]

    def is_valid_input(original: str, processed: str) -> bool:
        """True if `processed` is `original` with zero or more whole tokens swapped for '[REDACTED]'."""
        low = processed.lower()
        if any(marker in low for marker in _REFUSAL_MARKERS):
            return False
        orig, proc = _word_tokens(original), _word_tokens(processed)
        if len(orig) != len(proc):
            return False
        return all(p in (o, "[REDACTED]") for o, p in zip(orig, proc))

    def _leading_punct(word: str) -> str:
        i = 0
        while i < len(word) and word[i] in _PUNCT:
            i += 1
        return word[:i]

    def _trailing_punct(word: str) -> str:
        j = len(word)
        while j > 0 and word[j - 1] in _PUNCT:
            j -= 1
        return word[j:]

In [ ]:
if EVALUATION:
    # Alignment repair: force the model output back onto the original word sequence.

    def clean_llm_response(original_text: str, llm_response: str) -> str:
        """Re-align a model response to the original word-for-word, recovering from drift.

        Walks the original sentence by sentence in 5-word chunks; keeps response words that
        match (or are '[REDACTED]'), reverts the rest, and re-syncs when the response has
        slipped out of position.
        """
        original_normalized = normalize_whitespace(original_text)
        llm_normalized = normalize_whitespace(llm_response)
        original_words = original_normalized.split()
        llm_words = llm_normalized.split()

        if len(original_words) == len(llm_words) or len(original_words) == 0:
            return llm_normalized

        search_window = min(5, len(original_words))
        start_position = find_pattern_in_response(original_words[:search_window], llm_words)
        if start_position == -1:
            for i, w in enumerate(llm_words):
                if w == original_words[0] or _is_redacted_with_punct(w):
                    start_position = i
                    break
        if start_position == -1:
            return llm_normalized

        trimmed_response = llm_words[start_position:]
        cleaned_result = []
        resp_word_idx = 0

        for sentence in split_into_sentences(original_normalized):
            sentence_words = sentence.split()
            cleaned_sentence_words = []
            for chunk_start in range(0, len(sentence_words), 5):
                chunk_size = min(5, len(sentence_words) - chunk_start)
                orig_chunk = sentence_words[chunk_start:chunk_start + chunk_size]

                if resp_word_idx + chunk_size > len(trimmed_response):
                    cleaned_sentence_words.extend(orig_chunk)
                    break

                resp_chunk = trimmed_response[resp_word_idx:resp_word_idx + chunk_size]
                matches = sum(
                    resp_chunk[i] == orig_chunk[i] or _is_redacted_with_punct(resp_chunk[i])
                    for i in range(chunk_size)
                )
                mismatches = chunk_size - matches

                if mismatches == 0:
                    cleaned_sentence_words.extend(resp_chunk)
                    resp_word_idx += chunk_size
                elif mismatches <= 2 and (matches >= chunk_size / 2 or chunk_size <= 3):
                    cleaned_sentence_words.extend(
                        resp_chunk[i]
                        if (resp_chunk[i] == orig_chunk[i] or _is_redacted_with_punct(resp_chunk[i]))
                        else orig_chunk[i]
                        for i in range(chunk_size)
                    )
                    resp_word_idx += chunk_size
                else:
                    new_pos = find_pattern_in_response(orig_chunk, trimmed_response[resp_word_idx:])
                    if new_pos != -1:
                        resp_word_idx += new_pos
                        if resp_word_idx + chunk_size <= len(trimmed_response):
                            cleaned_sentence_words.extend(
                                trimmed_response[resp_word_idx:resp_word_idx + chunk_size]
                            )
                            resp_word_idx += chunk_size
                        else:
                            cleaned_sentence_words.extend(orig_chunk)
                    else:
                        cleaned_sentence_words.extend(orig_chunk)
            cleaned_result.append(" ".join(cleaned_sentence_words))

        return " ".join(cleaned_result)

    def enforce_word_count_match(
        original_text: str, processed_text: str, stats: dict | None = None
    ) -> str:
        """Force `processed` to `original`'s whitespace-token count, reverting non-redaction edits.

        A span the model collapsed to one '[REDACTED]' is re-expanded to one '[REDACTED]' per
        source word; when `stats` is given, each such expansion is counted there.
        """
        original_words = original_text.split()
        processed_words = processed_text.split()
        if original_words == processed_words:
            return processed_text

        matcher = difflib.SequenceMatcher(a=original_words, b=processed_words, autojunk=False)
        corrected_words = []
        for tag, i1, i2, j1, j2 in matcher.get_opcodes():
            orig_chunk = original_words[i1:i2]
            proc_chunk = processed_words[j1:j2]

            if tag == "equal":
                corrected_words.extend(orig_chunk)
            elif tag == "replace" and len(orig_chunk) == len(proc_chunk):
                corrected_words.extend(
                    proc_word
                    if (_is_redacted_with_punct(proc_word) or proc_word == orig_word)
                    else orig_word
                    for orig_word, proc_word in zip(orig_chunk, proc_chunk)
                )
            elif tag == "replace" and any(_is_redacted_with_punct(w) for w in proc_chunk):
                expanded = ["[REDACTED]"] * len(orig_chunk)
                expanded[0] = _leading_punct(orig_chunk[0]) + "[REDACTED]"
                expanded[-1] = expanded[-1] + _trailing_punct(orig_chunk[-1])
                corrected_words.extend(expanded)
                if stats is not None and len(orig_chunk) > 1:
                    stats["redacted_span_expansions"] += 1
                    stats["redacted_extra_tokens"] += len(orig_chunk) - 1
            elif tag in ("replace", "delete"):
                corrected_words.extend(orig_chunk)
            # tag == "insert" -> drop

        return " ".join(corrected_words)

In [ ]:
if EVALUATION:
    # Tiered fallback (chunk -> line -> word), ported from scripts/de_identify_steplab.py.
    # On any validation failure the text is retried at a finer granularity; the last
    # resort is the original text unchanged (never leak PII by emitting a bad edit).

    def process_word_by_word(original_text, prompt, api_call_func, stats):
        """Redact one whitespace token at a time; keep the original token on any failure."""
        processed_words = []
        for word in original_text.split():
            try:
                response = api_call_func(word, prompt)
                if response == "API_ERROR":
                    processed_words.append(word)
                    stats["word_failures"] += 1
                    continue
                enforced = enforce_word_count_match(word, clean_llm_response(word, response), stats)
                if not is_valid_input(word, enforced):
                    processed_words.append(word)
                    stats["word_failures"] += 1
                else:
                    parts = enforced.split()
                    processed_words.append(parts[0] if parts else word)
            except Exception:
                processed_words.append(word)
                stats["word_failures"] += 1
        return " ".join(processed_words)

    def process_line_with_fallback(line_text, prompt, api_call_func, stats):
        """Redact a single line; fall back to word-by-word if the line-level result is invalid."""
        line = line_text.strip()
        if not line:
            return line_text

        def _word_fallback():
            stats["line_failures"] += 1
            enforced = enforce_word_count_match(
                line, process_word_by_word(line, prompt, api_call_func, stats), stats
            )
            return enforced if is_valid_input(line, enforced) else line

        try:
            response = api_call_func(line, prompt)
            if response == "API_ERROR":
                return _word_fallback()
            enforced = enforce_word_count_match(line, clean_llm_response(line, response), stats)
            if is_valid_input(line, enforced):
                return enforced
            return _word_fallback()
        except Exception:
            try:
                return _word_fallback()
            except Exception:
                stats["word_failures"] += len(line.split())
                return line

    def process_chunk_with_fallback(chunk_text, prompt, api_call_func, stats):
        """Redact a multi-sentence chunk; fall back to line-by-line if the chunk result is invalid."""

        def _line_fallback():
            stats["chunk_failures"] += 1
            return "\n".join(
                process_line_with_fallback(line, prompt, api_call_func, stats) if line.strip() else line
                for line in chunk_text.split("\n")
            )

        try:
            response = api_call_func(chunk_text, prompt)
            if response == "API_ERROR":
                return _line_fallback()
            enforced = enforce_word_count_match(chunk_text, clean_llm_response(chunk_text, response), stats)
            if is_valid_input(chunk_text, enforced):
                return enforced
            return _line_fallback()
        except Exception:
            return _line_fallback()

    def process_text_content(original_text, prompt, api_call_func, stats):
        """Redact a full transcript body chunk by chunk."""
        return " ".join(
            process_chunk_with_fallback(chunk, prompt, api_call_func, stats)
            for chunk in split_into_sentences(original_text)
        )

In [ ]:
if EVALUATION:
    # Local stand-in for de_identify_steplab.py's openrouter_api_call: same signature,
    # same "Text to edit: " prefix, same "API_ERROR" sentinel on failure.

    def make_local_api_call(gen_model):
        """Return an api_call_func(post_text, prompt) that runs a greedy local generation."""

        def local_api_call(post_text, prompt):
            try:
                messages = [
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": "Text to edit: " + post_text},
                ]
                text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                inputs = tokenizer(text, return_tensors="pt").to(gen_model.device)
                with torch.no_grad():
                    output_ids = gen_model.generate(
                        **inputs,
                        max_new_tokens=inputs["input_ids"].shape[1] + 64,
                        do_sample=False,
                    )
                completion = output_ids[0][inputs["input_ids"].shape[1]:]
                return tokenizer.decode(completion, skip_special_tokens=True).strip()
            except Exception as e:
                print(f"[LOCAL MODEL ERROR] {e}")
                return "API_ERROR"

        return local_api_call

    def run_harness_on_file(file_path, prompt, api_call_func, stats):
        """Redact one transcript file: split header, normalize speakers, redact body, reassemble."""
        with open(file_path, "r", encoding="utf-8") as fh:
            all_lines = fh.readlines()

        if len(all_lines) == 0:
            return ""
        if len(all_lines) == 1:
            return remove_empty_speaker_lines(normalize_speaker_format(all_lines[0]))

        has_header = not _is_speaker_line(all_lines[0])
        if has_header:
            first_line = normalize_whitespace(all_lines[0].rstrip("\n"))
            remaining_content = "".join(all_lines[1:]).strip()
        else:
            first_line = None
            remaining_content = "".join(all_lines).strip()

        remaining_content = remove_empty_speaker_lines(normalize_speaker_format(remaining_content))
        if not remaining_content:
            return first_line if has_header else ""

        processed = process_text_content(remaining_content, prompt, api_call_func, stats)
        processed = remove_empty_speaker_lines(normalize_speaker_format(processed))

        return f"{first_line}\n\n{processed}" if has_header else processed

### Run the harness

Redact every `evaluation_data/original/*.txt` transcript with the base model and every adapter
in `ADAPTER_SPECS`. `harness_file_stats.csv` records per-file fallback counts and how many
multi-word `[REDACTED]` spans the harness had to expand.

In [ ]:
if EVALUATION:
    EVAL_ORIGINAL_DIR = "/kaggle/input/datasets/jopopapi/pii-stf-dataset/original"
    EVAL_PROMPTS_CSV = "/kaggle/input/datasets/jopopapi/pii-stf-dataset/prompts.csv"
    EVAL_PROMPT_LABEL = "prompt 3"
    EVAL_OUTPUT_DIR = "/kaggle/working/eval_outputs"

    # Counters threaded through the fallback pipeline. redacted_span_expansions /
    # redacted_extra_tokens track how often (and by how many tokens) the harness had to
    # expand a model's single "[REDACTED]" into one-per-word to keep the word count.
    STAT_FIELDS = [
        "chunk_failures", "line_failures", "word_failures",
        "redacted_span_expansions", "redacted_extra_tokens",
    ]

    def _new_stats():
        return dict.fromkeys(STAT_FIELDS, 0)

    with open(EVAL_PROMPTS_CSV, "r", encoding="utf-8", newline="") as f:
        prompt_text = next(
            (row["prompt"] for row in csv.DictReader(f) if row["label"] == EVAL_PROMPT_LABEL), None
        )
    if prompt_text is None:
        raise ValueError(f"No prompt labeled {EVAL_PROMPT_LABEL!r} found in {EVAL_PROMPTS_CSV}")

    eval_files = sorted(f for f in os.listdir(EVAL_ORIGINAL_DIR) if f.endswith(".txt"))

    # "base" and every adapter share one model object: disable_adapter() / use_adapter()
    # switch which (if any) LoRA layers are active for the duration of the `with` block below.
    if sanity_model is not None and adapter_labels:
        run_specs = [("base", sanity_model, sanity_model.disable_adapter)]
        run_specs += [
            (label, sanity_model, functools.partial(use_adapter, sanity_model, label))
            for label in adapter_labels
        ]
    else:
        run_specs = [("base", sanity_base, contextlib.nullcontext)]

    harness_outputs = {}
    harness_stats = {}
    harness_file_stats = []
    for run_label, gen_model, adapter_ctx in run_specs:
        gen_model.eval()
        outputs = {}
        run_stats = _new_stats()
        out_dir = os.path.join(EVAL_OUTPUT_DIR, run_label)
        os.makedirs(out_dir, exist_ok=True)
        with adapter_ctx():
            api_call_func = make_local_api_call(gen_model)
            for filename in eval_files:
                file_stats = _new_stats()
                redacted_text = run_harness_on_file(
                    os.path.join(EVAL_ORIGINAL_DIR, filename), prompt_text, api_call_func, file_stats
                )
                outputs[filename] = redacted_text
                with open(os.path.join(out_dir, filename), "w", encoding="utf-8") as fh:
                    fh.write(redacted_text + "\n")
                for k in STAT_FIELDS:
                    run_stats[k] += file_stats[k]
                harness_file_stats.append({"model": run_label, "filename": filename, **file_stats})
                print(
                    f"[{run_label}] {filename} "
                    f"(span_expansions={file_stats['redacted_span_expansions']})"
                )
        harness_outputs[run_label] = outputs
        harness_stats[run_label] = run_stats
        print(f"[{run_label}] " + " ".join(f"{k}={run_stats[k]}" for k in STAT_FIELDS))

    os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)
    stats_csv_path = os.path.join(EVAL_OUTPUT_DIR, "harness_stats.csv")
    with open(stats_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["model", *STAT_FIELDS])
        writer.writeheader()
        for run_label, run_stats in harness_stats.items():
            writer.writerow({"model": run_label, **run_stats})

    file_stats_csv_path = os.path.join(EVAL_OUTPUT_DIR, "harness_file_stats.csv")
    pd.DataFrame(harness_file_stats, columns=["model", "filename", *STAT_FIELDS]).to_csv(
        file_stats_csv_path, index=False
    )

    labels_joined = "|".join(label for label, _, _ in run_specs)
    print(f"\nSaved redacted transcripts to {EVAL_OUTPUT_DIR}/<{labels_joined}>/")
    print(f"Saved per-run stats to {stats_csv_path} and per-file stats to {file_stats_csv_path}")

### Score against the human gold

Gold and prediction are both projected onto the original transcript with difflib before labeling, so the gold's single `[REDACTED]` per span and the harness's one-per-word expansion line up. The gold files are only read, never modified.

In [ ]:
if EVALUATION:
    # Token-level scoring, ported from scripts/transcript_evaluator.py (confusion-matrix
    # tp/tn/fp/fn, per-file + corpus aggregation).
    #
    # The gold files collapse a multi-word PII span to a single "[REDACTED]", while the
    # harness expands it to one "[REDACTED]" per source word. To reconcile the two we
    # project BOTH onto the original transcript (their shared anchor) with difflib, so
    # gold and prediction end up on the same per-original-word index space. The gold
    # files themselves are only read, never modified.

    def is_redacted_token(tok: str) -> bool:
        """True if the whitespace token contains a '[REDACTED]' marker."""
        return bool(tok.strip()) and REDACT_PATTERN.search(tok) is not None

    def tokenize(text):
        return (text or "").split()

    def count_redacted_tokens(text):
        if text is None:
            return float("nan")
        return float(sum(is_redacted_token(t) for t in tokenize(text)))

    def project_redaction_labels(original_tokens, other_tokens):
        """Per-original-token 0/1 labels: 1 where `other` redacted or dropped that word."""
        labels = [0] * len(original_tokens)
        sm = difflib.SequenceMatcher(a=original_tokens, b=other_tokens, autojunk=False)
        for tag, i1, i2, j1, j2 in sm.get_opcodes():
            if tag == "equal":
                for k in range(i1, i2):
                    if is_redacted_token(other_tokens[j1 + (k - i1)]):
                        labels[k] = 1
            elif tag == "delete":
                for k in range(i1, i2):
                    labels[k] = 1
            elif tag == "replace" and any(is_redacted_token(t) for t in other_tokens[j1:j2]):
                for k in range(i1, i2):
                    labels[k] = 1
        return labels

    def align_labels(a_labels, b_labels):
        """Defensive length match -- project_redaction_labels already returns equal lengths."""
        n = min(len(a_labels), len(b_labels))
        return a_labels[:n], b_labels[:n]

    def _precision_recall(tp: int, fp: int, fn: int) -> tuple[float, float]:
        """Precision/recall with well-defined edge cases instead of NaN.

        No PII in the gold (human_pos == 0) is common (many transcripts have none) and is
        not an undefined case: recall is vacuously 1.0 (nothing to miss), and precision is
        1.0 if the model also redacted nothing, 0.0 if it redacted something that wasn't
        PII. NaN is reserved for the one genuinely undefined case: PII existed but the
        model made zero predictions at all.
        """
        human_pos, pred_pos = tp + fn, tp + fp
        if human_pos == 0:
            return (1.0 if pred_pos == 0 else 0.0), 1.0
        prec = (tp / pred_pos) if pred_pos > 0 else float("nan")
        return prec, tp / human_pos

    def compute_metrics(y_true, y_pred):
        if len(y_true) == 0:
            return {"accuracy": None, "precision": None, "recall": None, "kappa": None,
                    "tp": 0, "tn": 0, "fp": 0, "fn": 0, "total_words": 0}
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        prec, rec = _precision_recall(tp, fp, fn)
        if len(set(y_true)) == 1 and len(set(y_pred)) == 1:
            kappa = 1.0 if y_true[0] == y_pred[0] else 0.0
        else:
            try:
                kappa = cohen_kappa_score(y_true, y_pred)
            except Exception:
                kappa = 0.0
        return {"accuracy": accuracy_score(y_true, y_pred), "precision": prec, "recall": rec,
                "kappa": kappa, "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
                "total_words": int(len(y_true))}

    def compute_corpus_metrics_sklearn(all_true, all_pred):
        if all_true.size == 0:
            return {"corpus_accuracy": float("nan"), "corpus_precision": float("nan"),
                    "corpus_recall": float("nan"), "corpus_kappa": float("nan"),
                    "sum_tp": 0, "sum_tn": 0, "sum_fp": 0, "sum_fn": 0, "sum_total_words": 0}
        tn, fp, fn, tp = confusion_matrix(all_true, all_pred, labels=[0, 1]).ravel()
        prec, rec = _precision_recall(tp, fp, fn)
        return {"corpus_accuracy": accuracy_score(all_true, all_pred),
                "corpus_precision": prec, "corpus_recall": rec,
                "corpus_kappa": cohen_kappa_score(all_true, all_pred),
                "sum_tp": int(tp), "sum_tn": int(tn), "sum_fp": int(fp), "sum_fn": int(fn),
                "sum_total_words": int(all_true.size)}

    EVAL_REDACTED_DIR = "/kaggle/input/datasets/jopopapi/pii-stf-dataset/redacted"
    RESULTS_CSV_PATH = os.path.join(EVAL_OUTPUT_DIR, "eval_results.csv")
    SUMMARY_CSV_PATH = os.path.join(EVAL_OUTPUT_DIR, "eval_summary.csv")

    expansion_by_key = {(r["model"], r["filename"]): r for r in harness_file_stats}

    per_file_rows = []
    summary_rows = []
    for run_label, _, _ in run_specs:
        all_true, all_pred = [], []
        for filename, ai_text in harness_outputs[run_label].items():
            redacted_path = os.path.join(EVAL_REDACTED_DIR, filename)
            original_path = os.path.join(EVAL_ORIGINAL_DIR, filename)
            if not (os.path.exists(redacted_path) and os.path.exists(original_path)):
                continue
            with open(redacted_path, "r", encoding="utf-8") as f:
                human_text = f.read()
            with open(original_path, "r", encoding="utf-8") as f:
                original_text = f.read()

            original_tokens = tokenize(original_text)
            y_true, y_pred = align_labels(
                project_redaction_labels(original_tokens, tokenize(human_text)),
                project_redaction_labels(original_tokens, tokenize(ai_text)),
            )
            all_true.extend(y_true)
            all_pred.extend(y_pred)

            m = compute_metrics(y_true, y_pred)
            fstats = expansion_by_key.get((run_label, filename), {})
            per_file_rows.append({
                "filename": filename, "model": run_label, "prompt_label": EVAL_PROMPT_LABEL,
                "accuracy": m["accuracy"], "precision": m["precision"], "recall": m["recall"],
                "kappa": m["kappa"], "tp": m["tp"], "tn": m["tn"], "fp": m["fp"], "fn": m["fn"],
                "total_words": m["total_words"],
                "human_redacted_count": count_redacted_tokens(human_text),
                "ai_redacted_count": count_redacted_tokens(ai_text),
                "ai_redacted_span_expansions": fstats.get("redacted_span_expansions", 0),
                "ai_redacted_extra_tokens": fstats.get("redacted_extra_tokens", 0),
            })

        all_true = np.array(all_true, dtype=int)
        all_pred = np.array(all_pred, dtype=int)
        corpus = compute_corpus_metrics_sklearn(all_true, all_pred)

        file_metrics = [r for r in per_file_rows if r["model"] == run_label]
        summary_rows.append({
            "model": run_label, "prompt_label": EVAL_PROMPT_LABEL,
            "files_evaluated": len(file_metrics),
            "avg_accuracy": np.nanmean([r["accuracy"] for r in file_metrics]) if file_metrics else float("nan"),
            "avg_precision": np.nanmean([r["precision"] for r in file_metrics]) if file_metrics else float("nan"),
            "avg_recall": np.nanmean([r["recall"] for r in file_metrics]) if file_metrics else float("nan"),
            "avg_kappa": np.nanmean([r["kappa"] for r in file_metrics]) if file_metrics else float("nan"),
            "redacted_span_expansions": harness_stats[run_label]["redacted_span_expansions"],
            "redacted_extra_tokens": harness_stats[run_label]["redacted_extra_tokens"],
            **corpus,
        })
        print(
            f"[{run_label}] files={len(file_metrics)} sum_total_words={corpus['sum_total_words']} "
            f"corpus_accuracy={corpus['corpus_accuracy']:.4f} corpus_precision={corpus['corpus_precision']:.4f} "
            f"corpus_recall={corpus['corpus_recall']:.4f} corpus_kappa={corpus['corpus_kappa']:.4f} "
            f"tp={corpus['sum_tp']} tn={corpus['sum_tn']} fp={corpus['sum_fp']} fn={corpus['sum_fn']}"
        )

    per_file_df = pd.DataFrame(per_file_rows, columns=[
        "filename", "model", "prompt_label", "accuracy", "precision", "recall", "kappa",
        "tp", "tn", "fp", "fn", "total_words",
        "human_redacted_count", "ai_redacted_count",
        "ai_redacted_span_expansions", "ai_redacted_extra_tokens",
    ])
    summary_df = pd.DataFrame(summary_rows, columns=[
        "model", "prompt_label", "files_evaluated", "avg_accuracy", "avg_precision", "avg_recall", "avg_kappa",
        "corpus_accuracy", "corpus_precision", "corpus_recall", "corpus_kappa",
        "sum_tp", "sum_tn", "sum_fp", "sum_fn", "sum_total_words",
        "redacted_span_expansions", "redacted_extra_tokens",
    ])

    per_file_df.to_csv(RESULTS_CSV_PATH, index=False)
    summary_df.to_csv(SUMMARY_CSV_PATH, index=False)

    print(f"\nSaved per-file metrics ({len(per_file_df)} rows) to {RESULTS_CSV_PATH}")
    print(f"Saved per-model summary to {SUMMARY_CSV_PATH}")
    display(summary_df)

### Compare to base

Each adapter's corpus-level metrics minus the base model's -- positive is an improvement,
negative is a regression.

In [ ]:
if EVALUATION:

    def _save_fig(fig, name: str) -> None:
        """Save `fig` as both PNG and SVG under EVAL_OUTPUT_DIR so Kaggle offers them for download."""
        for ext in ("png", "svg"):
            fig.savefig(os.path.join(EVAL_OUTPUT_DIR, f"{name}.{ext}"), bbox_inches="tight")

    _compare_cols = ["corpus_accuracy", "corpus_precision", "corpus_recall", "corpus_kappa"]
    _delta_cols = [f"delta_{c}" for c in _compare_cols]

    if "base" not in summary_df["model"].values:
        print("No 'base' row in summary_df -- skipping base comparison.")
        comparison_df = pd.DataFrame(columns=["model", *_compare_cols, *_delta_cols])
    else:
        base_row = summary_df.loc[summary_df["model"] == "base", _compare_cols].iloc[0]
        comparison_df = summary_df.loc[summary_df["model"] != "base", ["model"] + _compare_cols].copy()
        for col in _compare_cols:
            comparison_df[f"delta_{col}"] = comparison_df[col] - base_row[col]
        comparison_df = comparison_df.reset_index(drop=True)

    COMPARISON_CSV_PATH = os.path.join(EVAL_OUTPUT_DIR, "eval_comparison.csv")
    comparison_df.to_csv(COMPARISON_CSV_PATH, index=False)
    print(f"Saved base-vs-adapter comparison ({len(comparison_df)} rows) to {COMPARISON_CSV_PATH}")

    if not comparison_df.empty:
        _display_df = comparison_df[["model", *_delta_cols]].copy()
        for col in _delta_cols:
            _display_df[col] = _display_df[col].map(lambda x: f"{x:+.4f}")
        display(_display_df)

        ax = comparison_df.set_index("model")[_delta_cols].plot(
            kind="bar", figsize=(max(8, len(comparison_df)), 5)
        )
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_ylabel("Δ vs base")
        ax.set_title(f"Adapter change vs base model ({EVAL_PROMPT_LABEL})")
        ax.legend([c.removeprefix("delta_corpus_") for c in _delta_cols], loc="best")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        _save_fig(ax.figure, "eval_comparison_deltas")
        plt.show()
    else:
        print("No adapters to compare against base.")

### Visualize results

Corpus-level precision/recall/accuracy/kappa per model, plus how often each model needed the
span-expansion fallback -- a quick way to compare the base model against every adapter.

In [ ]:
if EVALUATION:
    # _save_fig is defined in the "Compare to base" cell above.
    _metric_cols = ["corpus_accuracy", "corpus_precision", "corpus_recall", "corpus_kappa"]
    ax = summary_df.set_index("model")[_metric_cols].plot(kind="bar", figsize=(max(8, len(summary_df)), 5), ylim=(0, 1))
    ax.set_ylabel("Score")
    ax.set_title(f"Redaction quality by model ({EVAL_PROMPT_LABEL})")
    ax.legend(loc="lower right")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    _save_fig(ax.figure, "eval_summary_metrics")
    plt.show()

    ax = summary_df.set_index("model")["redacted_span_expansions"].plot(
        kind="bar", figsize=(max(8, len(summary_df)), 4)
    )
    ax.set_ylabel("Multi-word spans expanded")
    ax.set_title("How often the harness fell back to one-[REDACTED]-per-word")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    _save_fig(ax.figure, "eval_summary_expansions")
    plt.show()

    print(f"Saved plots (PNG + SVG) to {EVAL_OUTPUT_DIR}/eval_summary_metrics.* and eval_summary_expansions.*")